<div align="right"><a href="S05-Frequency-response-and-sweep-measurement.ipynb">Sprache wechseln zu Englisch 🇬🇧</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

### Aufgabe: Frequenzgang und Sweep-Messung

Wir betrachten ein System mit der folgenden Z-Transformationsfunktion:

$$
H(z) = 1 - z^{-m}
$$

**a)** Berechnen Sie den Frequenz-, Phasen- und Impulsantwort des Systems.  
(Mit Stift und Papier)  
(Machen Sie es wirklich — eine ähnliche Aufgabe könnte in der mündlichen Prüfung vorkommen.)

**b)** Berechnen Sie den Amplituden- und Phasengang mit der MATLAB-Funktion `freqz()` und stellen Sie die Ergebnisse (in linearer Skalierung) für den Bereich  
$0 \leq \Omega < 2 \pi$ dar.  
Setzen Sie $m = 1$.  
Überprüfen Sie Ihre Berechnungen aus Teil a) dieser Aufgabe.

**c)** Berechnen Sie die Impulsantwort mit dem Befehl `impz()` und plotten Sie die Ergebnisse.  
Erstellen Sie ein Nullstellen-Pol-Diagramm mit der Funktion `zplane()`.  
Erklären Sie, um welchen Filtertyp es sich handelt.

**d)** In der Praxis kann ein Frequenzgang oft mit sogenannten Sweep-Messungen ermittelt werden.  
Bei Sweep-Messungen wird ein System mit Schwingungen unterschiedlicher Frequenzen angeregt.  
Ein LTI-System reagiert auf eine Schwingung mit derselben Frequenz, jedoch mit unterschiedlicher Phase und Amplitude.  
Wenn die Frequenz über den interessierenden Teil des Spektrums ($0 \dots \pi$) variiert wird,  
können der Amplituden- und Phasengang bestimmt werden.

In dieser Aufgabe soll der Amplitudengang mit einer Anregung der Form

$$
x(k) = \cos(\Omega_i k)
$$

gemessen werden.  
Vergleichen Sie die Ergebnisse mit denen aus Teil b).

**e)** Regen Sie das System mit einer Sprungfolge $\varepsilon(k)$ an und zeigen Sie, dass die erste Differenz der Sprungantwort der Impulsantwort entspricht.  
Hinweis: Verwenden Sie den Befehl `diff()`.

In [ ]:
# BEGIN SOLUTION
H = [1,-1] ; % Impulsantwort = Polynom H(z) = Zaehlerpolynom (H(z)) fuer FIR 
N = 32;      % Anzahl der Werte die ausgegeben werden sollen

% #####  Teilaufgabe b:  Amplituden- und Phasengang  #####

[y,x] = freqz(H,1,256,'whole') ;

% # grafische Ausgabe # %
% Amplitudengang
figure;
plot(x/(2*pi),abs(y));
title('Amplitudengang'); 
xlabel('\Omega/2\pi'); 
ylabel('Amplitude');

% Phasengang
figure;
plot(x/(2*pi),-angle(y)/pi*180) ; % Achtung: -angle() wg. Phasengangdefinition
set(gca,'YTick',[-90,-45,0,45,90]);
title('Phasengang'); 
xlabel('\Omega/2\pi'); 
ylabel('Phase in Grad');

% #####  Teilaufgabe c:  Impulsantwort, Pol- Nullstellendiagramm  #####

h  = impz(H,1,4); % 5 Abtastwerte
lh = length(h);

% # grafische Ausgabe # %
% Impulsantwort
figure;
stem(0:lh-1,h); 
axis([-0.1 3.1 -1 1]);
hold on; 
plot([-0.1,3.1],[0,0],'k'); 
hold off;
title('Impulsantwort'); 
xlabel('k'); 
ylabel('Amplitude') ;

% Pol- und Nullstellen in der z-Ebene
figure;
zplane(H,1);
title('Pol-Nullstellendiagramm');

% #####  Teilaufgabe d:  Wobbelmessung - Kosinuserregung #####
lf = length(H) ;
k_wobbel = (0:N-1+2*(lf-1));    % Zum Wobbeln Sequenz der Laenge N+2*(lf-1)
n_wobbel = 0:N/2-1;             % normierte Frequenzen
erg = [];

for n = n_wobbel
    wobbel = cos(2*pi*n/N*k_wobbel) ;                 % Eigentliche Wobblemessung
    foo = conv(H,wobbel);                          % Es wird gewobbelt
    erg = [erg;max(abs(foo(1+(lf-1):length(foo)-(lf-1))))]; % ohne Einschwing-vorgaenge
end

[c,d] = freqz(H,1,N);  % Vergleich Wobbel - Amplitudengang

% # grafische Ausgabe # %
figure;
plot(2*n_wobbel/N,erg,'r+'); 
hold on;
plot(d/pi,abs(c),'b-') 
hold off;
title('+ Wobblemessung, - theoret. Amplitudengang'); 
xlabel('\Omega/\pi'); 
ylabel('Amplitude');

% #####  Teilaufgabe e:  Wobbelmessung - Exponentialerregung #####
erg = [];
for n = n_wobbel
    wobbel = exp(j*2*pi*n/N*k_wobbel);                 % Eigentliche Wobblemessung
    foo = conv(H,wobbel);                              % Es wird gewobbelt
    erg = [erg; max(abs(foo(1+(lf-1):length(foo)-(lf-1))))]; % ohne Einschwing-
                                                             % vorgaenge
end

% # grafische Ausgabe % #
figure;
plot(2*n_wobbel/N,erg,'r+');
hold on;
plot(d/pi,abs(c),'b-') 
hold off;
title('+ Wobblemessung, - theoret. Amplitudengang'); 
xlabel('\Omega/\pi'); 
ylabel('Amplitude');

% #####  Teilaufgabe f:  Sprungantwort  #####

epsilon = [zeros(10,1); ones(10,1)];
sprungantwort = conv(h,epsilon);
hdiff=diff(sprungantwort);

% # grafische Ausgabe # %
% Sprungfolge
figure;
stem((0:length(epsilon)-1)-10,epsilon);
title('Sprungfolge'); 
xlabel('k'); 
ylabel('\epsilon(k)');

% Sprungantwort
figure;
stem((0:length(sprungantwort)-1)-10,sprungantwort);
hold on; 
plot([-10,15],[0,0],'k'); 
hold off;
title('Sprungantwort'); 
xlabel('k'); 
ylabel('Amplitude');

% Impulsantwort
figure;
stem((0:length(hdiff)-1)-10,hdiff);
hold on; 
plot([-10,15],[0,0],'k'); 
hold off;
title('Impulsantwort'); 
xlabel('k'); 
ylabel('Amplitude');
# END SOLUTION

### Lösung: Frequenzgang und Sweep-Messung

**a)**

$$
H(z) = 1 - z^{-m}
$$

Die **Amplitudenantwort** $\left|H(e^{j \Omega})\right|$ des Systems ergibt sich zu:

$$
H(e^{j \Omega}) = 1 - e^{-j \Omega m}
$$

$$
\begin{aligned}
\left|H(e^{j \Omega})\right|
  &= \left| 1 - e^{-j \Omega m} \right| \\
  &= \left|  e^{-j \frac{\Omega}{2} m} \left( e^{j \frac{\Omega}{2} m} - e^{-j  \frac{\Omega}{2} m} \right) \right| \\
  &= \underbrace{\left|  e^{-j \frac{\Omega}{2} m} \right|}_{1} \left|\left( e^{j \frac{\Omega}{2} m} - e^{-j \frac{\Omega}{2} m} \right) \right| \\
  &= \left| \left( e^{j \frac{\Omega}{2} m} - e^{-j \frac{\Omega}{2} m} \right) \right| \\
  &= \left| 2 j \sin \left( \frac{\Omega}{2} m\right) \right| \\
\left|H(e^{j \Omega})\right|  &= 2 \left| \sin \left( \frac{\Omega}{2} m\right) \right|
\end{aligned}
$$

Die **Phasenantwort** $b(\Omega)$ ergibt sich gemäß der Definition: Achtung: Die Phasenantwort ist hier mit negativem Vorzeichen definiert, sodass sich die Gruppenlaufzeit durch positive Ableitung der Phasenfunktion ergibt  
$\tau_g(\Omega) = \frac{\mathrm{d}\, b(\Omega)}{\mathrm{d}\, \Omega}$  
Normalerweise wird die Phasenantwort positiv definiert, wodurch zur Bestimmung der Gruppenlaufzeit ein negatives Vorzeichen nötig ist.

$$
H(e^{j \Omega}) = \left| H(e^{j \Omega}) \right| e^{- j b(\Omega) }
$$

und wird berechnet durch:

$$
\begin{aligned}
b(\Omega) &= - \arg\!\left(H(e^{j\Omega})\right) \\
          &= - \arctan\!\left(\frac{\Im\{H(e^{j\Omega})\}}{\Re\{H(e^{j\Omega})\}}\right)
\end{aligned}
$$

Dazu ist es zweckmäßig, die Übertragungsfunktion analog zur obigen Rechnung umzuformen:

$$
\begin{aligned}
H(e^{j \Omega}) &= 1 - e^{-j \Omega m} \\
                &= e^{-j \frac{\Omega}{2} m}  \left[ e^{j \frac{\Omega}{2} m} - e^{-j \frac{\Omega}{2} m}\right] \\
                &= \left[ \cos \left( \frac{\Omega}{2} m \right) - j \sin \left( \frac{\Omega}{2} m \right) \right]
                   \left[ 2 j \sin \left( \frac{\Omega}{2} m \right) \right] \\
                &= 2 \sin^2 \left( \frac{\Omega}{2} m \right) + j\, 2 \sin \left( \frac{\Omega}{2} m \right)
                   \cos \left( \frac{\Omega}{2} m \right)
\end{aligned}
$$

$$
\begin{aligned}
b(\Omega) &= - \arctan \frac{\Im\{H(e^{j\Omega})\}}{\Re\{H(e^{j\Omega})\}} \\
          &= - \arctan \frac{ 2 \sin \left( \tfrac{\Omega}{2} m \right)
        \cos \left( \tfrac{\Omega}{2} m \right) }{ 2 \sin^2 \left( \tfrac{\Omega}{2} m \right)  } \\
          &= - \arctan \frac{ \cos \left( \tfrac{\Omega}{2} m \right) }{ \sin \left( \tfrac{\Omega}{2} m \right)  } \\
          &= - \arctan \cot \left( \tfrac{\Omega}{2} m \right)  \\
          & \downarrow \quad \text{mit } \arctan x = \tfrac{\pi}{2} - \operatorname{arccot} x \\
      &= - \tfrac{\pi}{2} + \operatorname{arccot}\, \cot \left( \tfrac{\Omega}{2} m \right)  \\
b(\Omega) &= - \tfrac{\pi}{2} + \tfrac{\Omega}{2} m
\end{aligned}
$$


**b)**

Wird die MATLAB-Funktion `freqz()` verwendet, so werden Amplituden- und Phasengang gemäß den folgenden Abbildungen dargestellt:

<img src="./../pngs/ldsv2a18_b1.png" width="250">  
<img src="./../pngs/ldsv2a18_b2.png" width="250">  

Es zeigt sich, dass der Kurvenverlauf mit den analytischen Ergebnissen aus Teil a) übereinstimmt.

**c)**

Die Impulsantwort, die mit der Funktion `impz()` erzeugt wird, ist in den folgenden Abbildungen dargestellt.  
Ihr Verlauf ergibt sich direkt aus der Z-Transformationsfunktion.

$$
h(k) = 1 \delta_0(k) - 1 \delta_0(k-m)
$$

<img src="./../pngs/ldsv2a18_c1.png" width="250">  
<img src="./../pngs/ldsv2a18_c2.png" width="250">  

Der Pol im Ursprung des Pol-Nullstellen-Diagramms ergibt sich aus

$$
H(z) = 1 - z^{-m} = \frac{z^m - 1}{z^m}
$$

Es handelt sich um einen **Hochpassfilter**, da die niedrigen Frequenzen durch die Nullstellen an der Position $z_0 = 1$ unterdrückt werden.

**d)**

Der Amplitudengang, der mit Hilfe einer Sweep-Messung ermittelt wurde, stimmt mit der theoretischen Berechnung überein.

<img src="./../pngs/ldsv2a18_d.png" width="250">  

**e)**

Die folgende Abbildung zeigt die Sprungantwort des Systems.  
Neben der tatsächlichen Antwort ist auch die Transientenantwort zu erkennen, die durch die endliche Länge der Sprungfolge $\epsilon(k)$ entsteht.

<img src="./../pngs/ldsv2a18_e1.png" width="250">  
<img src="./../pngs/ldsv2a18_e2.png" width="250">  

Durch Bildung der ersten Differenz kann die Impulsantwort des Systems erneut beobachtet werden.

<img src="./../pngs/ldsv2a18_e3.png" width="250">  